In [ ]:
# 1.0 Imports
from os import listdir, getcwd
from os.path import isfile, join
import csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.base import clone
from scipy.interpolate import interp1d
from scipy.signal import argrelextrema, find_peaks, savgol_filter
from scipy.ndimage import minimum_filter1d
from scipy.integrate import trapezoid

#models
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import VotingClassifier


In [ ]:
# 2.0 Functions
def import_paths(res= "High", exo= "7"):
    """

    # Function to batch import filepaths of spectra
    # This is reliant on the specific folder structure:
        >Dir
            >Renamed Data
                >{res} Res
                    >MCF{exo}_U/F_date

    # ---------- #
    res - string, resolution of Raman spectrometer as "High" or "Low"
    exo - string, exosome type used on chip as "7", "10a" or "10aPS"
    # ---------- #
    
    """
    
    root_dir = join(getcwd(), "Renamed Data", f"{res} Res")
    files = [
        join(root_dir, folder, file)
        for folder in listdir(root_dir)
        if folder.startswith(f"MCF{exo}_")
        for file in listdir(join(root_dir, folder))
    ]

    return files

def txtr_to_df(filepath, metad= False):
    """

    # Function to import .txtr files to a dataframe, specifically for the format of the low res Raman spec
    # Each line is delimited by a semicolon (below). 
    # I'm assuming all file formats are the same, or at least header rows. Haven't checked yet, only guaranteed to work on a sample, but it should work
    # use is literally |  df = txtr_to_df(filepath)  |
    # for metadata | df, metadata = txtr_to_df(filepath, metad=True)  |

    # ---------- #
    filepath - string, input filepath
    metad - bool, do you want metadata returned as a secondary output. Not sure what you'd use it for besides laser power, but worth having the whole thing.
    # ---------- #

    """
    
    metadata = {}
    data_start = None
    with open(filepath, 'r') as f:
        lines = f.readlines()
    for i, line in enumerate(lines):
        line = line.strip()
        if line.startswith("Pixel;"):
            data_start = i
            break
        if ";" in line:
            key, _, value = line.partition(";")
            metadata[key.strip()] = value.strip()
    df = pd.read_csv(filepath, sep=";", skiprows=data_start)
    df = df.dropna(axis=1)
    df.columns = df.columns.str.strip()
    df = df[["Raman Shift", "Dark Subtracted #1"]].copy()
    df.columns = ["Raman Shift", "Intensity"]
    df = df[df["Raman Shift"] >= 400]
    df = df[df["Raman Shift"] <= 1800].reset_index(drop= True)
    df["Index"] = np.arange(len(df))

    if metad:
        return df, metadata
    else:
        return df

def txt_to_df(filepath, delimiter= "\\t"):
    """

    # Function to import .txtfiles to a dataframe, specific for format of high res Raman spec
    # Discards headers and manually defines them.
    # Specific to high res, not functionalising for anything else
    # There is no metadata to scrape

    # ---------- #
    filepath - string input filepath
    delimiter - string input in-line delimiter for .txt file (multicolumn)
    # ---------- #


    """
    
    df = pd.read_csv(filepath, sep=delimiter, header= None, engine= "python")
    df = df.dropna(axis=1)
    cols =  ["Raman Shift", "Intensity"]
    df.columns = cols
    df["Index"] = np.arange(len(df))

    return df

def batch_import(res= "High", exo= "7"):
    """

    # Function to batch import files, essentially master function for this section
    # Calls the 3 above functions. Might be worth just defining them internally instead, but the txt(r) imports might be useful outside

    # ---------- #
    res - string, as in import_paths()
    exo - string, as in import_paths()
    # ---------- #
    
    """
    filenames = import_paths(res, exo)
    if res == "Low":
        dfs = [txtr_to_df(file) for file in filenames]
    elif res == "High":
        dfs = [txt_to_df(file) for file in filenames]
    else:
        raise ValueError(f"Mode was set to {mode}\nThis is not a supported filetype")

    return dfs

def baseline_minmax(df, n_splits= 10, trough_mode= 0, trough_param_mode = 0):
    """

    # Baseline correction and minmax normalisation to 0 - 1
    # I know it's not versatile, I don't particularly care

    # ---------- #
    df - pandas dataframe, your dataframe after data correction (above)
    n_splits - int, number of splits for baseline (1/%dataframe per split)
    trough_mode - int:
        0 - argrelextrema - derivative based. standard approach for general signal processing
        1 - thresholded minima - local minima, minimum prominence and minimum distance between minima to threshold
        2 - percentile based minima. NOT RECOMMENDED for this type - good for flat plots, curved will break
        3 - morphological - erosion using min filter
        4 - argrelextrema with savitzky-golay smoothing
    trough_params (wip) - pass parameters like prominence, window_length etc. NOT IN USE
    # ---------- #

    """
    
    trough_dict = {
        # Dummy Argrel
        "Argrel": [
            [0],
            [0]
        ],
    
        "Threshold": [
            [0.05, 10],
            [0.15, 20]
        ],

        "Percentile": [
            [5],
            [5]
        ],
    
        "Morpho": [
            [10],
            [20]
        ],
    
        "Argrel_SavGol": [
            [6, 3],
            [60, 9]
        ]
    }
    
    troughs = None
    shift = df.set_index("Index")["Raman Shift"]
    intensity = df.set_index("Index")["Intensity"]
    
    if trough_mode == 0:
        trough_params= trough_dict["Argrel"][trough_param_mode]
        troughs = argrelextrema(intensity.values, np.less)[0]
    elif trough_mode == 1:
        trough_params= trough_dict["Threshold"][trough_param_mode]
        troughs, props = find_peaks(-intensity.values, prominence=trough_params[0], distance=trough_params[1])
    elif trough_mode == 2:
        trough_params= trough_dict["Percentile"][trough_param_mode]
        threshold = np.percentile(intensity.values, trough_params[0])
        troughs = np.where(intensity.values <= threshold)[0]
    elif trough_mode == 3:
        trough_params= trough_dict["Morpho"][trough_param_mode]
        baseline_candidate = minimum_filter1d(intensity.values, size=trough_params[0])
        troughs = np.where(intensity.values == baseline_candidate)[0]
    elif trough_mode == 4:
        trough_params= trough_dict["Argrel_SavGol"][trough_param_mode]
        intensity_smooth = savgol_filter(intensity.values, window_length=trough_params[0], polyorder=trough_params[1])
        troughs = argrelextrema(intensity_smooth, np.less)[0]
    else:
        raise ValueError(f"Invalid trough_mode {trough_mode}.")

    splits = np.array_split(troughs, n_splits)
    baselines = [s.min() for s in splits]
    baselines.append(troughs[-1])
    X_curve = shift.loc[baselines].values
    Y_curve = intensity.loc[baselines].values
    X_troughs = shift.loc[troughs].values
    Y_troughs = intensity.loc[troughs].values
    curve = interp1d(X_curve, Y_curve, kind='linear', fill_value='extrapolate')
    baseline_at_troughs = curve(X_troughs)
    uncorrected = Y_troughs - baseline_at_troughs
    corrected = np.clip(uncorrected, 0, None)
    normalised = MinMaxScaler().fit_transform(np.array(np.array(corrected).reshape(-1, 1))).flatten()

    return X_troughs, normalised
def all_preprocesses_batch(dfs, res= "High"):
    """

    # Returns a list:
        >per preprocessing technique
            >per df
                >X values
                >Y values

    # ---------- #
    dfs - list, containing pandas dataframes structured ["Raman Shift", "Intensity", "Index"]
    res - string, resolution of spectrometer used for dataset
    # ---------- #
    
    """
    if res == "High":
        extracted = [[baseline_minmax(df, n_splits= 20, trough_mode= i, trough_param_mode= 1) for df in dfs] for i in range(0, 5)]
    elif res == "Low":
        extracted = [[baseline_minmax(df, n_splits= 20, trough_mode= 0, trough_param_mode= 0) for df in dfs] for i in range(0, 5)]
    else:
        raise ValueError(f"res was set to {res}\nThis is not a supported resolution.\nChoose either High or Low")

    return extracted

def extreme_outlier_removal(dataset, max_thresh, min_thresh):
    """

    # Identifies outliers outside a specific threshold (otherwise interpolate_at_x() breaks)
    # Basically error management. There are like 4 spectra that are off by about 50 cm-1 somehow

    # ---------- #
    dataset - array, all_preprocesses[x][y]
    max_thresh - int, maximum threshold
    min_thresh - int, minimum theshold
    # ---------- #
    
    """
    
    keep = []
    kept_indices = []
    for idx, i in enumerate(dataset):
        x = i[0]
        if x.max() >= max_thresh and x.min() <= min_thresh:
            keep.append(i)
            kept_indices.append(idx)
        else:
            print(f"Outlier detected and removed: max={x.max()}, min={x.min()}")
    return keep, kept_indices

def interpolate_at_x(x, y, x_target):
    """

    # Mathematical function to identify intersection of x-value with plot of points either side of it
    # Outputs intersection, target

    # ---------- #
    x - array, x-values
    y - array, y-values
    x_target - int, target value
    # ---------- #
    
    """
    sort_idx = np.argsort(x)
    x = x[sort_idx]
    y = y[sort_idx]
    idx = np.searchsorted(x, x_target)
    if idx == 0 or idx == len(x):
        raise ValueError("Target x-value is outside data range.")
    x1, y1 = x[idx-1], y[idx-1]
    x2, y2 = x[idx], y[idx]

    return y1 + (y2 - y1) * ((x_target - x1) / (x2 - x1)), x_target

def batch_interpolate_at_x(x, y, x_targets):
    """

    # Runs interpolate_at_x over an array
    # Outputs interpolate_at_x as a list

    # ---------- #
    x - array, x-values
    y - array, y-values
    x_targets - array, indexes for interpolation
    # ---------- #
    
    """
    
    intersections = [interpolate_at_x(x, y, target) for target in x_targets]
    return intersections


def find_area(x, y, intersections):
    """

    # Finds the area of a range over a single dataset (below, generate_auc_df)
    # Outputs a list, containing [0] left bound [1] right bound [2] area

    # ---------- #
    x - array, x-values
    y - array, y-values
    x_targets - array, output of batch_interpolate_at_x
    # ---------- #
    
    """
    
    outputs = []
    for i in range(len(intersections)):
        left_lim = intersections[i][1]
        left_value = intersections[i][0]
        try:
            right_lim = intersections[i+1][1]
            right_value = intersections[i+1][0]
        except:
            ValueError()
        if right_lim != left_lim:
            mask = (x > left_lim) & (x < right_lim)
            x_vals = np.concatenate((
                [left_lim],
                x[mask],
                [right_lim]
            ))
            y_vals = np.concatenate((
                [left_value],
                y[mask],
                [right_value]
            ))
            outputs.append([left_lim, right_lim, trapezoid(y_vals, x_vals)])

    return outputs

def generate_auc_df(dataset, range_width= 5, class_col= True, class_val = 1):
    """

    # Generates a dataframe, based on interpolate_at_x, batch_interpolate_at_x and find_area that is explicitly labeled
    # Outputs a single row dataframe
    # Range reduced to 430 - 1780 due to differences in wavenumber in different datasets.

    # ---------- #
    dataset - array, "Layer 4" index of all_preprocesses in __main__
    range_width - int, width of a "band", best as a multiple of 5, I didn't add error management
    class_col - bool, if yes adds a column "Class"
    class_val - int, the value in class_col
    # ---------- #
    
    """
    
    x = dataset[0]
    y = dataset[1]
    intersections = batch_interpolate_at_x(x, y, np.arange(430, 1780, range_width))
    data = find_area(x, y, intersections)
    frame = pd.DataFrame([[i[-1] for i in data]], columns=[f"{i[0]} - {i[1]}" for i in data])
    if class_col:
        frame["Class"] = class_val

    return frame

def batch_auc_df(dataset, range_width= 5, class_col= True, class_val= 1):
    """

    # Runs generate_auc_df() over a series of files (per-res&exo combo)
    # Outputs a dataframe of all returns, with class column if used.

    # ---------- #
    dataset - array, all_preprocesses[x][x]
    range_width - as in generate_auc_df()
    class_col - as in generate_auc_df()
    class_val - as in generate_auc_df()
    # ---------- #
    
    """
    
    individual_dfs = [generate_auc_df(i, range_width, class_col, class_val) for i in dataset]
    df = pd.concat(individual_dfs).reset_index(drop= True)
    return df

In [ ]:
# 3.0 Main
if __name__ == "__main__":
    model_dict = {
        "DecisionTree": 
        {
            "Model": DecisionTreeClassifier(random_state= 42, class_weight= "balanced"),
            "ParamGrid": {
                "criterion": ["gini", "entropy"],
                "max_depth": [None, 3, 5, 8, 12],
                "min_samples_split": [2, 5, 10],
                "min_samples_leaf": [1, 2, 4]
            }
        },
        "RandomForest":
        {
            "Model": RandomForestClassifier(bootstrap= True, n_jobs= -1, random_state= 42, class_weight= "balanced"),
            "ParamGrid": {
                "n_estimators": [100, 200, 500, 800],
                "max_depth": [None, 5, 10, 20],
                "min_samples_split": [2, 5 ,10],
                "min_samples_leaf": [1, 2, 4],
                "max_features": ["sqrt", "log2", 0.5]
            }
        },
        "KNearest":
        {
            "Model": KNeighborsClassifier(algorithm= "auto", n_jobs= -1),
            "ParamGrid": {
                "n_neighbors": [3, 5, 11, 21],
                "weights": ["uniform", "distance"],
                "p": [1, 2]
            }
        },
        "LGBoost":
        {
            "Model": LGBMClassifier(n_jobs= -1, class_weight= "balanced"),
            "ParamGrid": {
                "boosting_type": ["gbdt", "dart"],
                "num_leaves": [31, 63, 127, 255],
                "max_depth": [-1, 6, 10, 15],
                "learning_rate": [0.05, 0.1, 0.2],
                "n_estimators": [100, 200, 500, 1000],
                "min_child_samples": [10, 20, 40],
                "subsample": [0.7, 0.9, 1.0],
                "colsample_bytree": [0.7, 0.9, 1.0],
                "reg_alpha": [0, 1, 5],
                "reg_lambda": [0, 1, 5]
            }
        },
        "XGBoost":
        {
            "Model": XGBClassifier(booster= "gbtree", n_jobs= -1, random_state= 42, tree_method= "hist", eval_metric= "logloss", objective="binary:logistic"),
            "ParamGrid": {
                "n_estimators": [200, 300, 600, 900],
                "learning_rate": [0.05, 0.1, 0.2],
                "max_depth": [3, 5, 7]
            }
        },
        "AdaBoost":
        {
            "Model": AdaBoostClassifier(estimator=DecisionTreeClassifier(class_weight="balanced"), random_state= 42),
            "ParamGrid": {
                "n_estimators": [50, 100, 200],
                "learning_rate": [0.1, 0.5, 1.0]
            }
        },
        "SupportVector":
        {
            "Model": SVC(random_state= 42, class_weight= "balanced", probability= True),
            "ParamGrid": {
                "C": [0.1, 1, 10],
                "kernel": ["rbf", "poly", "linear"],
                "gamma": ["scale", "auto"]
            }
        }
    }
    
    alldfs = {
        "High Pos": batch_import("High", "7"),
        "High Neg": batch_import("High", "10a"),
        "Low Pos": batch_import("Low", "7"),
        "Low Neg": batch_import("Low", "10a"),
        "Low Poly": batch_import("Low", "10aPS")
    }

    model_names = [
        "DecisionTree",
        "RandomForest",
        "KNearest",
        "LGBoost",
        "XGBoost",
        "AdaBoost",
        "SupportVector"
    ]

    # Import Filepaths
    hr_pos_paths = import_paths("High", "7")
    hr_neg_paths = import_paths("High", "10a")
    lr_pos_paths = import_paths("Low", "7")
    lr_neg_paths = import_paths("Low", "10a")
    lr_poly_paths = import_paths("Low", "10aPS")

    # Preprocess Data
    all_preprocesses = [all_preprocesses_batch(alldfs[key], key.split(" ")[0]) for key in alldfs]

    # Outlier Removal
    hr_pos, hr_pos_idx = extreme_outlier_removal(all_preprocesses[0][0], 1775, 425)
    hr_neg, hr_neg_idx = extreme_outlier_removal(all_preprocesses[1][0], 1775, 425)
    lr_pos, lr_pos_idx = extreme_outlier_removal(all_preprocesses[2][0], 1775, 425)
    lr_neg, lr_neg_idx = extreme_outlier_removal(all_preprocesses[3][0], 1775, 425)
    lr_poly, lr_poly_idx = extreme_outlier_removal(all_preprocesses[4][0], 1775, 425)
    
    # Set DataFrames with Indeces
    hrpos_df = batch_auc_df(hr_pos, 5, True, 1)
    hrneg_df = batch_auc_df(hr_neg, 5, True, 0)
    lrpos_df = batch_auc_df(lr_pos, 5, True, 1)
    lrneg_df = batch_auc_df(lr_neg, 5, True, 0)
    lrpoly_df = batch_auc_df(lr_poly, 5, True, 0)

    # Add Traceback Column Before Splits
    hrpos_df["source"] = "High_Pos"
    hrneg_df["source"] = "High_Neg"
    lrpos_df["source"] = "Low_Pos"
    lrneg_df["source"] = "Low_Neg"
    lrpoly_df["source"] = "Low_Poly"

    Full = pd.concat([hrpos_df, hrneg_df, lrpos_df, lrneg_df, lrpoly_df], axis=0)



In [ ]:
Full.head()
print(Full.iloc[0])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

row = Full.iloc[-1]

# Drop non-numeric metadata columns if present
numeric_row = row.drop(['Class', 'source'], errors='ignore').astype(float)

fig, ax = plt.subplots(figsize=(18, 5))

x = np.arange(len(numeric_row))
ax.bar(x, numeric_row.values, width=1.0, color='steelblue', edgecolor='none')

# Label every 50th band
tick_step = 50
ax.set_xticks(x[::tick_step])
ax.set_xticklabels(numeric_row.index[::tick_step], rotation=45, ha='right', fontsize=8)

ax.set_xlabel('Wavelength band (nm)')
ax.set_ylabel('Reflectance')
ax.set_title(f'Sample Spectrum (Class: {row.get("Class", "N/A")}, Source: {row.get("source", "N/A")})')
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(18, 5))

ax.plot(lr_poly[-1][0], lr_poly[-1][1])

ax.set_xlabel('Wavelength band (nm)')
ax.set_ylabel('Reflectance')
ax.set_title(f'Sample Spectrum (Class: {row.get("Class", "N/A")}, Source: {row.get("source", "N/A")})')
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
hr_pos_SavGol = extreme_outlier_removal(all_preprocesses[0][-1], 1775, 425)
fig, ax = plt.subplots(figsize=(18, 5))

ax.plot(hr_pos_SavGol[0][0], hr_pos_SavGol[0][1])

ax.set_xlabel('Wavelength band (nm)')
ax.set_ylabel('Reflectance')
ax.set_title(f'Sample Spectrum (Class: {row.get("Class", "N/A")}, Source: {row.get("source", "N/A")})')
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
filenames = import_paths()
dfs = [txt_to_df(file) for file in filenames]

fig, ax = plt.subplots(figsize=(18, 5))

ax.plot(dfs[-1]["Raman Shift"], dfs[-1]["Intensity"])

ax.set_xlabel('Wavelength band (nm)')
ax.set_ylabel('Reflectance')
ax.set_title(f'Sample Spectrum (Class: {row.get("Class", "N/A")}, Source: {row.get("source", "N/A")})')
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Find which indices survived outlier removal
def extreme_outlier_removal_indexed(dataset, max_thresh, min_thresh):
    keep = []
    kept_indices = []
    for idx, i in enumerate(dataset):
        x = i[0]
        if x.max() >= max_thresh and x.min() <= min_thresh:
            keep.append(i)
            kept_indices.append(idx)
        else:
            print(f"Outlier removed at index {idx}: max={x.max():.1f}, min={x.min():.1f}")
    return keep, kept_indices

# Re-run outlier removal with index tracking
lr_poly_clean, lr_poly_kept_idx = extreme_outlier_removal_indexed(all_preprocesses[4][0], 1775, 425)

# Pick Sample
sample_pos = 15
raw_idx = lr_poly_kept_idx[sample_pos]
raw_df  = alldfs["Low Poly"][raw_idx]
pre     = lr_poly_clean[sample_pos]
auc_row = lrpoly_df.iloc[raw_idx].drop(['Class', 'source'], errors='ignore').astype(float)

# --- Plot ---
fig, axes = plt.subplots(3, 1, figsize=(18, 12))

axes[0].plot(raw_df["Raman Shift"].values, raw_df["Intensity"].values, color='steelblue', linewidth=0.8)
axes[0].set_title("Sample Spectrum (Class: 0, Source: Low_Neg)")

axes[1].plot(pre[0], pre[1], color='steelblue', linewidth=0.8)
axes[1].set_title("Sample Spectrum (Class: 0, Source: Low_Neg)")

x2 = np.arange(len(auc_row))
axes[2].bar(x2, auc_row.values, width=1.0, color='steelblue', edgecolor='none')
axes[2].set_xticks(x2[::50])
axes[2].set_xticklabels(auc_row.index[::50], rotation=45, ha='right', fontsize=8)
axes[2].set_title("Sample Spectrum (Class: 0, Source: Low_Neg)")

for ax in axes:
    ax.set_xlabel('Raman Shift (cm⁻¹)')
    ax.set_ylabel('Intensity')
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle(f'Sample comparison — Low Poly index {sample_pos} (raw file {raw_idx})', y=1.01)
plt.tight_layout()
plt.savefig("Negative_Class_Sample_Comparison.png", dpi=500, bbox_inches="tight")


In [ ]:
# Find which indices survived outlier removal
def extreme_outlier_removal_indexed(dataset, max_thresh, min_thresh):
    keep = []
    kept_indices = []
    for idx, i in enumerate(dataset):
        x = i[0]
        if x.max() >= max_thresh and x.min() <= min_thresh:
            keep.append(i)
            kept_indices.append(idx)
        else:
            print(f"Outlier removed at index {idx}: max={x.max():.1f}, min={x.min():.1f}")
    return keep, kept_indices

# Re-run outlier removal with index tracking
hr_pos_clean, hr_pos_kept_idx = extreme_outlier_removal_indexed(all_preprocesses[0][0], 1775, 425)

# Pick sample 
sample_pos = 15
raw_idx = hr_pos_kept_idx[sample_pos]
raw_df  = alldfs["High Pos"][raw_idx]
pre     = hr_pos_clean[sample_pos]
auc_row = hrpos_df.iloc[raw_idx].drop(['Class', 'source'], errors='ignore').astype(float)

# Plot
fig, axes = plt.subplots(3, 1, figsize=(18, 12))

axes[0].plot(raw_df["Raman Shift"].values, raw_df["Intensity"].values, color='steelblue', linewidth=0.8)
axes[0].set_title("Sample Spectrum (Class: 1, Source: high_Pos)")

axes[1].plot(pre[0], pre[1], color='steelblue', linewidth=0.8)
axes[1].set_title("Sample Spectrum (Class: 1, Source: High_Pos)")

x2 = np.arange(len(auc_row))
axes[2].bar(x2, auc_row.values, width=1.0, color='steelblue', edgecolor='none')
axes[2].set_xticks(x2[::50])
axes[2].set_xticklabels(auc_row.index[::50], rotation=45, ha='right', fontsize=8)
axes[2].set_title("Sample Spectrum (Class: 1, Source: High_Pos)")

for ax in axes:
    ax.set_xlabel('Raman Shift (cm⁻¹)')
    ax.set_ylabel('Intensity')
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle(f'Sample comparison — High Positive index {sample_pos} (raw file {raw_idx})', y=1.01)
plt.tight_layout()
plt.savefig("Positive_Class_Sample_Comparison.png", dpi=300, bbox_inches="tight")


In [ ]:

features = ""
with open("Texts\\Feature_Importances_N30.txt", "r") as f:
    for line in f:
        features += line
#print(features)
per_model_features = []


model_features = {}
current_model = None

for line in features.splitlines():
    line = line.rstrip()
    if "— Top 30 Features:" in line:
        current_model = line.split("—")[0].strip()
        model_features[current_model] = []
        continue
    if "Feature importance not available" in line:
        current_model = None
        continue
    if current_model and "importance:" in line:
        parts = line.split()
        start = parts[1]
        end = parts[3]
        model_features[current_model].append(f"{start}-{end}")

for i in model_features:
    print(model_features[i])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

row = Full.iloc[15]

# Drop non-numeric metadata columns if present
numeric_row = row.drop(['Class', 'source'], errors='ignore').astype(float)

fig, ax = plt.subplots(figsize=(18, 5))

x = np.arange(len(numeric_row))
ax.bar(x, numeric_row.values, width=1.0, color='steelblue', edgecolor='none')
# Parse index bins once: "430 - 435" → (430, 435)
index_ranges = []
for label in numeric_row.index:
    start, end = label.split("-")
    index_ranges.append((int(start.strip()), int(end.strip())))

# Overlay feature ranges
for ranges in model_features.values():
    for r in ranges:
        r_start, r_end = r.split("-")
        r_start = int(r_start.strip())
        r_end = int(r_end.strip())

        # Find x positions covered by this feature range
        x_positions = [
            i for i, (idx_start, idx_end) in enumerate(index_ranges)
            if idx_start >= r_start and idx_end <= r_end
        ]

        if x_positions:
            ax.axvspan(
                min(x_positions),
                max(x_positions) + 1,
                color="red",
                alpha=0.1
            )
tick_step = 50
ax.set_xticks(x[::tick_step])
ax.set_xticklabels(numeric_row.index[::tick_step], rotation=45, ha='right', fontsize=8)

ax.set_xlabel('Wavelength band (cm-1)')
ax.set_ylabel('Intensity')
ax.set_title(f'Sample Spectrum (Class: {row.get("Class", "N/A")}, Source: {row.get("source", "N/A")}) with Overlayed "Key Features"')
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig("Positive_Class_Sample_Feature_Overlay.png", dpi=300, bbox_inches="tight")


In [ ]:
print(row, numeric_row)

In [ ]:
feature_counts = {}

for features in model_features.values():
    for f in features:
        feature_counts[f] = feature_counts.get(f, 0) + 1
def feature_sort_key(f):
    return int(f.split("-")[0].strip())
for f in sorted(feature_counts, key=feature_sort_key):
    print(f, feature_counts[f])
    # n=5

In [ ]:
feature_counts = {}

for features in model_features.values():
    for f in features:
        feature_counts[f] = feature_counts.get(f, 0) + 1
def feature_sort_key(f):
    return int(f.split("-")[0].strip())
for f in sorted(feature_counts, key=feature_sort_key):
    print(f, feature_counts[f])
    #n=30

In [ ]:
# Overlayed Regions and Key Raman Regions
import matplotlib.pyplot as plt
import numpy as np
def band_midpoint(label):
    lo, hi = label.split('-')
    return (float(lo) + float(hi)) / 2

# Drop non-numeric metadata columns if present
numeric_row = row.drop(['Class', 'source'], errors='ignore').astype(float)

fig, ax = plt.subplots(figsize=(18, 5))

x = np.arange(len(numeric_row))
ax.bar(x, numeric_row.values, width=1.0, color='steelblue', edgecolor='none')

# Label every 50th band
tick_step = 50
ax.set_xticks(x[::tick_step])
ax.set_xticklabels(numeric_row.index[::tick_step], rotation=45, ha='right', fontsize=8)

wavelengths = np.array([band_midpoint(lbl) for lbl in numeric_row.index])
row = Full.iloc[0]
bands = [
    (680, 705),
    (995, 1005),
    (1695, 1720)
]
bands1 = [
    (640, 655),
    (680, 705),
    (790, 805),
    (995, 1005),
    (1030, 1040),
    (1375, 1385),
    (1580, 1610),
    (1695, 1720)
]

for start, end in bands1:
    i0 = np.searchsorted(wavelengths, start)
    i1 = np.searchsorted(wavelengths, end)

    ax.axvspan(i0, i1, color='gray', alpha=0.5, zorder=0)
    ax.text((i0 + i1) / 2,
    y_top,
    "Key Region",
    ha='center',
    va='bottom',
    rotation=45,
    fontsize=12
)
# Labeled Regions
labelled_bands = {
    "Cholesterol, Glycogen": {
        "range": [(430, 610)],
        "color": "gold"
    },
    "Tyrosine, Phenylalanine": {
        "range": [(610, 680)],
        "color": "royalblue"
    },
    "Nucleic Acids": {
        "range": [(680, 810)],
        "color": "limegreen"
    },
    "Proteins, Carbohydrates": {
        "range": [(830, 975)],
        "color": "cornflowerblue"
    },
    "Phenylalanine, Collagen": {
        "range": [(990, 1015)],
        "color": "orangered"
    },
    "Amide III, Lipids,\nCystosine, Guaninine": {
        "range": [(1140, 1260)],
        "color": "violet"
    },
    "Nucleic Acids": {
        "range": [(1290, 1330)],
        "color": "yellow"
    },
    "Tryptophan, Proteins": {
        "range": [(1340, 1380)],
        "color": "aquamarine"
    },
    "Proteins, Lipids": {
        "range": [(1395, 1500)],
        "color": "aqua"
    },
    "Amide I, Lipids": {
        "range": [(1520, 1700)],
        "color": "orange"
    }
}

y_top = ax.get_ylim()[1]

y_top = ax.get_ylim()[1]

for label, info in labelled_bands.items():
    for start, end in info["range"]:
        i0 = np.searchsorted(wavelengths, start)
        i1 = np.searchsorted(wavelengths, end)

        ax.axvspan(
            i0, i1,
            color=info["color"],
            alpha=0.25,
            zorder=0,
            label=label
        )

ax.set_xlabel('Wavelength band (cm -1)')
ax.set_ylabel('Reflectance')
ax.set_title(f'Sample Spectrum (Class: {row.get("Class", "N/A")}, Source: {row.get("source", "N/A")}) with Overlay of Spectral Regions and All Identified Feature Bands', y= 1.4)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(loc= "right")
#plt.tight_layout()
plt.show()